# Geospatial Research Radar - 03: optional web/social index bridge

This notebook handles the part that open scholarly APIs cannot cover well: public posts and publication pages indexed from LinkedIn, ResearchGate, personal sites, GitHub, and the wider web.

It intentionally does **not** automate a logged-in LinkedIn browser or scrape Google Scholar. Instead, it defines a tiny interchange format (`web_hits.json`) that a sanctioned search API, RSS collector, or external scheduled job can populate. The next morning, notebook 01 merges those hits into the same scoring pipeline.

In [ ]:
from radar_core import DEFAULT_CONFIG, deep_copy_jsonable, load_json, save_json, make_external_search_queries

config = load_json('radar_config.json', default=deep_copy_jsonable(DEFAULT_CONFIG))
queries = make_external_search_queries(config)
save_json(queries, 'web_search_queries.json')

print(f"Generated {len(queries)} external search queries.")
for q in queries[:20]:
    print('-', q['query'])

## `web_hits.json` interchange format

Your external search layer only needs to produce a JSON list like this. `title` and `url` are required; all other fields are optional.

In [ ]:
template = [
  {
    "title": "Example public post or publication title",
    "url": "https://example.org/item",
    "snippet": "A short snippet describing the method, dataset, or code release.",
    "date": "2026-09-04",
    "author": "Example Researcher",
    "source": "Web index",
    "open_access": True
  }
]

save_json(template, 'web_hits_TEMPLATE.json')
print('Saved web_hits_TEMPLATE.json')
print('
Do not rename the template to web_hits.json unless it contains real search results; notebook 01 automatically ingests web_hits.json when present.')

## Recommended Phase 2

Use a small server-side job (GitHub Actions, a scheduled cloud function, or cron) to run a web search provider each morning and write `web_hits.json`. That server-side bridge solves two browser-only JupyterLite limitations at once: CORS and unattended scheduling. The scholarly/OpenAlex portion can remain exactly as-is.